In [129]:
import pandas as pd
import numpy as np

In [130]:
avaliacoes = 'csv/base/avaliacoes.csv'
avaliacoes = pd.read_csv(avaliacoes, sep=',')
avaliacoes.head()

,id_avaliacao,id_reserva,nota_geral,nota_limpeza,nota_atendimento,nota_custo_beneficio,comentario,data_avaliacao
0,1,280,-3,7.2,8.6,8.1,Staff muito atencioso.,2023-07-13
1,2,312,9.4,5.7,9.8,5.4,Custo-benefício muito bom.,2024-05-21
2,3,727,8.1,5.4,7.7,8.4,"Ótima estadia, recomendo!",2024-05-20
3,4,964,8.6,7.4,5.9,8.6,"Ótima estadia, recomendo!",2023-04-17
4,5,726,8.5,7.8,6.0,7.4,Staff muito atencioso.,2023-06-26


In [131]:
reservas = 'csv/base/reservas.csv'
reservas = pd.read_csv(reservas, sep=',')
reservas['id_reserva'] = reservas['id_reserva'].astype(str)
reservas.head()

reservas['status_reserva'] = reservas['status_reserva'].str.upper().replace('CANCELADA', 'CANCELADO')
reservas['status_reserva'].value_counts()

status_reserva
CONFIRMADA     968
OVERBOOKING     18
CANCELADO        8
NO-SHOW          6
Name: count, dtype: int64

In [132]:
# nota geral como str ao invés de float (alguns valores estão como 'bom' e outros como número negativo)
# nota_custo_beneficio está com o mesmo problema da nota_geral

avaliacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 685 entries, 0 to 684
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_avaliacao          685 non-null    int64  
 1   id_reserva            685 non-null    int64  
 2   nota_geral            685 non-null    str    
 3   nota_limpeza          681 non-null    float64
 4   nota_atendimento      683 non-null    float64
 5   nota_custo_beneficio  685 non-null    str    
 6   comentario            632 non-null    str    
 7   data_avaliacao        685 non-null    str    
dtypes: float64(2), int64(2), str(4)
memory usage: 42.9 KB


In [133]:
reservas.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id_reserva      1000 non-null   str    
 1   id_hospede      1000 non-null   int64  
 2   id_quarto       1000 non-null   str    
 3   id_hotel        1000 non-null   int64  
 4   data_checkin    1000 non-null   str    
 5   data_checkout   1000 non-null   str    
 6   canal_reserva   995 non-null    str    
 7   valor_diaria    997 non-null    float64
 8   status_reserva  1000 non-null   str    
 9   data_reserva    1000 non-null   str    
dtypes: float64(1), int64(2), str(7)
memory usage: 78.3 KB


In [134]:
# nota_limpeza e nota_atendimento contém dados nulos, é preciso decidir se os dados serão excluídos da análise, feito a média ou se será mantido o valor 0
# Preenche dados nulos da coluna comentarios
avaliacoes['comentario'] = avaliacoes['comentario'].fillna('Sem comentário')
avaliacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 685 entries, 0 to 684
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id_avaliacao          685 non-null    int64  
 1   id_reserva            685 non-null    int64  
 2   nota_geral            685 non-null    str    
 3   nota_limpeza          681 non-null    float64
 4   nota_atendimento      683 non-null    float64
 5   nota_custo_beneficio  685 non-null    str    
 6   comentario            685 non-null    str    
 7   data_avaliacao        685 non-null    str    
dtypes: float64(2), int64(2), str(4)
memory usage: 42.9 KB


In [135]:
#ajustes na coluna nota_geral

avaliacoes['nota_geral'] = avaliacoes['nota_geral'].replace('bom', None)
avaliacoes['nota_geral'] = avaliacoes['nota_geral'].str.replace(',', '.').astype(float)

In [136]:
#ajustes na coluna nota_geral

avaliacoes['errados_menos_que_0'] = avaliacoes['nota_geral'] < 0
avaliacoes.loc[avaliacoes['errados_menos_que_0'], 'nota_geral'] = 0
avaliacoes['nota_geral'] = avaliacoes['nota_geral'].replace('errados_menos_que_0', 0)

avaliacoes['errados_maior_que_10'] = avaliacoes['nota_geral'] > 10
avaliacoes.loc[avaliacoes['errados_maior_que_10'], 'nota_geral'] = 10
avaliacoes['nota_geral'] = avaliacoes['nota_geral'].replace('errados_maior_que_10', 10)

In [ ]:
#ajustes na coluna nota_limpeza

avaliacoes['errados_negativos'] = avaliacoes['nota_limpeza'] < 0
avaliacoes.loc[avaliacoes['errados_negativos'], 'nota_limpeza'] = 0
avaliacoes['nota_limpeza'] = avaliacoes['nota_limpeza'].replace('errados_negativos', 0)

avaliacoes['errados_superiores_a_10'] = avaliacoes['nota_limpeza'] > 10
avaliacoes.loc[avaliacoes['errados_superiores_a_10'], 'nota_limpeza'] = 10
avaliacoes['nota_limpeza'] = avaliacoes['nota_limpeza'].replace('errados_superiores_a_10', 10)


avaliacoes['nota_limpeza']


0      7.2
1      5.7
2      5.4
3      7.4
4      7.8
      ... 
680    9.1
681    5.6
682    6.5
683    8.1
684    5.5
Name: nota_limpeza, Length: 685, dtype: float64

In [138]:
#ajustes na coluna nota_atendimento

avaliacoes['errados_neg'] = avaliacoes['nota_atendimento'] < 0
avaliacoes.loc[avaliacoes['errados_neg'], 'nota_atendimento'] = 0
avaliacoes['nota_atendimento'] = avaliacoes['nota_atendimento'].replace('errados_neg', 0)

avaliacoes['errados_super_10'] = avaliacoes['nota_atendimento'] > 10
avaliacoes.loc[avaliacoes['errados_super_10'], 'nota_atendimento'] = 10
avaliacoes['nota_atendimento'] = avaliacoes['nota_atendimento'].replace('errados_super_10', 10)
avaliacoes['nota_atendimento']

0      8.6
1      9.8
2      7.7
3      5.9
4      6.0
      ... 
680    8.4
681    5.6
682    8.5
683    6.3
684    9.8
Name: nota_atendimento, Length: 685, dtype: float64

In [139]:
#ajustes na coluna nota_custo_beneficio

avaliacoes['nota_custo_beneficio'] = avaliacoes['nota_custo_beneficio'].replace('muito bom', None)
avaliacoes['nota_custo_beneficio'] = avaliacoes['nota_custo_beneficio'].str.replace(',', '.').astype(float)

In [140]:
#correção avaliação antes do checkout



In [141]:
#ajuste id_reserva

avaliacoes['id_reserva'] = avaliacoes['id_reserva'].astype(str)

In [142]:
avaliacoes['']

KeyError: ''

In [ ]:
#ajuste id_reserva

df_cancelados = reservas[reservas['status_reserva'] == 'CANCELADO']
avaliacoes = avaliacoes[~avaliacoes['id_reserva'].isin(df_cancelados['id_reserva'])]


In [ ]:
#confirmação que deu certo
print(len(avaliacoes))
print(len(df_limpos))
print(len(df_cancelados))

677
677
8


In [ ]:
avaliacoes['id_avaliacao']

0        1
1        2
2        3
3        4
4        5
      ... 
672    673
673    674
674    675
675    676
676    677
Name: id_avaliacao, Length: 677, dtype: int64

In [143]:
avaliacoes.to_csv("./csv/tratado/avaliacoes_tratado.csv", index=False)